# 1. データセット収集 (Pi0 / VLAモデル向け)
Genesisシミュレータを利用して、エキスパート（ヒューリスティックまたは手動操作）のデモンストレーションを収集し、学習用データセットを作成


In [1]:
import genesis as gs
import numpy as np
import h5py
import os

# Genesisの初期化
gs.init(backend=gs.cuda) # CPU : gs.cpu , RTX GPUを利用する場合は gs.cuda に変更してください

scene = gs.Scene(show_viewer=False)
plane = scene.add_entity(gs.morphs.Plane())
robot = scene.add_entity(gs.morphs.UR5e(), randomize_dof_pos=True)
scene.build()

print('Genesis環境を構築しました。')


ModuleNotFoundError: No module named 'genesis'

In [ ]:
def collect_data(num_episodes=5, steps_per_episode=100):
    os.makedirs('dataset', exist_ok=True)
    with h5py.File('dataset/demo_data.hdf5', 'w') as f:
        for ep in range(num_episodes):
            ep_group = f.create_group(f'episode_{ep}')
            obs_list, action_list = [], []
            
            robot.set_dof_pos(np.zeros(robot.n_dofs))
            
            for step in range(steps_per_episode):
                # 仮のランダムアクション（実際は逆運動学等を用いたエキスパートポリシー）
                action = np.random.uniform(-0.1, 0.1, size=robot.n_dofs)
                robot.set_dof_target_vel(action)
                scene.step()
                
                obs_list.append(robot.get_dof_pos())
                action_list.append(action)
                
            ep_group.create_dataset('observations', data=np.array(obs_list))
            ep_group.create_dataset('actions', data=np.array(action_list))
    print('データ収集完了')

# 実行する場合はコメントアウト解除
# collect_data(num_episodes=2)
